# Shared model folds and scoring rules

Notebook 05 already saved the features. Do not rebuild them here. Create one shared **GroupKFold(5)** split by trip, with one row per GPS point, and reuse the saved fold file in every base model and the stacker.

The fold file includes flagged points and empty shortlists so point IDs stay complete. Training and scoring later exclude flagged points. Candidate misses stay in the Top-1 denominator.

This notebook does not train models or choose model hyperparameters.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold

output = Path("outputs/folds")
output.mkdir(parents=True, exist_ok=True)
seed = 42
points = pd.read_csv("outputs/candidates/points.csv", dtype={"trip_id": str})
folds = points[["trip_id", "group_id", "point_id"]].copy()
folds["fold"] = -1
for fold, (_, valid) in enumerate(GroupKFold(5).split(folds, groups=folds["trip_id"])):
    folds.loc[valid, "fold"] = fold
folds = folds.sort_values("group_id").reset_index(drop=True)
assert folds["group_id"].is_unique and folds["fold"].between(0, 4).all()
assert folds.groupby("trip_id")["fold"].nunique().eq(1).all()
folds.to_csv(output / "group_folds.csv", index=False)
folds[["trip_id", "fold"]].drop_duplicates().to_csv(output / "trip_folds.csv", index=False)

## Fold summary

GroupKFold is deterministic here (`shuffle=False`); the common seed is for model and tuning randomness. Its balancing unit is GPS points, not candidate rows. Each trip appears in exactly one validation fold.

The candidate radius and label thresholds were previously selected on all training trips. These folds therefore assess models conditional on that fixed preprocessing, rather than a fully nested evaluation of preprocessing choices.

In [2]:
points = points.merge(folds[["group_id", "fold"]], on="group_id", validate="one_to_one")
summary = points.groupby("fold").agg(
    trips=("trip_id", "nunique"), gps_points=("group_id", "size"), candidate_rows=("candidate_count", "sum"),
    flagged_points=("bad_gps", "sum")
)
summary["labelled_points"] = summary["gps_points"] - summary["flagged_points"]
summary["candidate_misses"] = points.assign(miss=points["true_arc_id"].ge(0) & points["true_in_candidates"].eq(False)).groupby("fold")["miss"].sum()
summary.to_csv(output / "summary.csv")
assert summary["trips"].sum() == 80 and len(summary) == 5
display(summary)

,trips,gps_points,candidate_rows,flagged_points,labelled_points,candidate_misses
fold,,,,,,
0,15,41747,3375095,1460,40287,693
1,16,41739,2053925,18,41721,1
2,16,41737,1603109,0,41737,0
3,16,41417,2407080,11,41406,0
4,17,41777,2239693,596,41181,70


## Shared OOF and metric contract

Save `oof_<model>.csv` with `group_id, segment_id, fold, score`. `score` is a per-point probability: normalize positive-class probabilities for logistic regression, KNN and the tree; apply stable per-point softmax to ranker scores. Keep `raw_score` as an optional extra column for diagnostics. All valid held-out candidates receive OOF predictions, including singleton and no-positive groups. Never train on a held-out trip.

Use candidate metadata for targets and the point file for the complete valid-point denominator. Top-1 includes misses as wrong. Log loss uses only groups with one positive candidate and is therefore conditional on candidate coverage. Report coverage alongside it.

Candidate-row precision, recall, macro F1 and the 2×2 confusion matrix use one argmax prediction per point. Precision and recall need not equal Top-1 when some candidate groups have no positive target.

Use the same saved folds for all four base-model OOF files. A stacker trained on those OOF inputs can be fit safely for final prediction. For an unbiased outer-CV stacker score, base predictions for meta-training must also be generated entirely inside that outer training split; merely applying CV to globally generated OOF inputs does not provide that guarantee.

KNN has no `random_state` argument. Fix input ordering and tie handling; use the seed for any training-group sampling. Imputation and scaling are fit inside each training split.

In [3]:
feature_columns = json.loads(Path("outputs/features/feature_columns.json").read_text())
contract = {
    "seed": seed,
    "fold_scheme": "GroupKFold(5), trip_id groups, shuffle=False, balanced by GPS-point count",
    "fold_file": "outputs/folds/group_folds.csv",
    "oof_required_columns": ["group_id", "segment_id", "fold", "score"],
    "oof_score": "per-point probability; sums to 1 within each group",
    "oof_optional_columns": ["raw_score"],
    "feature_columns": feature_columns + ["distance_ratio"],
    "distance_ratio": "(distance + 1 metre) / (distance - distance_above_nearest_m + 1 metre)",
    "top1_denominator": "all valid labelled points, including candidate misses and empty shortlists",
    "log_loss_denominator": "valid points whose candidate set has exactly one positive",
    "ranker_training": "at least 2 candidates and exactly 1 positive; recompute group sizes after splitting",
    "candidate_settings": json.loads(Path("outputs/candidates/settings.json").read_text()),
    "point_flag_threshold_m": float(pd.read_csv("outputs/labels/summary.csv")["threshold_m"].iloc[0]),
    "student_test_status": "raw trips available; labels, candidates and features not generated",
}
(output / "contract.json").write_text(json.dumps(contract, indent=2) + "\n")
print("Saved one fold mapping for", len(folds), "GPS points and", folds["trip_id"].nunique(), "trips.")

Saved one fold mapping for 208417 GPS points and 80 trips.
